# Claude full-text eligibility

Extract datasets with Claude and Gemini embeddings, then combine review results.

Method and inputs: [workflow guide](../../docs/workflow.md).

In [1]:
import sys
from pathlib import Path

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "notebooks" / "_shared.py").is_file():
        sys.path.insert(0, str(_parent / "notebooks"))
        break
else:
    raise FileNotFoundError("Start Jupyter inside the repository.")

from _shared import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index,
)

load_environment()

import csv
import os
import json
import pandas as pd
import re
import time

from collections import defaultdict, deque, Counter
from typing import List
from pydantic import BaseModel, Field

from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from google import genai
from anthropic import Anthropic


# Load environment variables
load_environment()
require_environment('CLAUDE_API_KEY', 'MODEL_CLAUDE', 'MODEL_SUPERVISOR_CLAUDE', 'MODEL_EMBEDDING', 'GEMINI_API_KEY')

MODEL= os.getenv("MODEL_CLAUDE")
MODEL_SUPERVISOR= os.getenv("MODEL_SUPERVISOR_CLAUDE")
MODEL_EMBEDDING = os.getenv("MODEL_EMBEDDING")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("CLAUDE_API_KEY")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
claude_client = Anthropic(api_key=ANTHROPIC_API_KEY)

BASE_DIR = str(project_root())
DOCUMENT_FOLDER = os.path.join(BASE_DIR, "data/CLAUDE_full_text_pdf")
VECTOR_DB_PATH = cache_directory("claude")

# The local cache is reused only when PDF contents and settings match.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[CLEANUP] Removed existent vectorstore: C:\Users\silve\GitHub\automating-prisma\vectorstore_claude\pdf_index


In [2]:
# Full-text PDFs stay in their configured local input directory.


In [2]:
# Embedding and text split configuration

CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
MAX_CONTEXT_CHARS = 30000

# Simple rate-limiting mechanism
_RATE_WINDOW_SEC = 60.0
_RATE_MAX_CALLS = 10
_rate_calls_log = deque()

_sent_calls_count = 0
_total_input_tokens = 0
_total_output_tokens = 0
_total_embedding_tokens = 0


def _rate_limit_block_until_allowed():
    """Blocks execution to respect Gemini API rate limits."""
    while True:
        now = time.monotonic()

        while _rate_calls_log and (now - _rate_calls_log[0]) >= _RATE_WINDOW_SEC:
            _rate_calls_log.popleft()

        if len(_rate_calls_log) < _RATE_MAX_CALLS:
            _rate_calls_log.append(now)
            return

        time.sleep(0.5)


# Pydantic response schemas
class ExtractedDatasets(BaseModel):
    datasets: List[str] = Field(description="List of official AQA dataset names found")

class HarmonizedSources(BaseModel):
    sources: List[str] = Field(description="Harmonized list of dataset names in input order")


def truncate_context(text: str, limit_chars: int) -> str:
    """Truncates long text while preserving character encoding."""
    if len(text) <= limit_chars:
        return text
    return text[:limit_chars - 3] + "..."

def clean_text(text: str) -> str:
    """Removes invalid UTF-8 characters."""
    return text.encode("utf-8", "ignore").decode("utf-8", "ignore")


# FAISS Index Construction
def build_or_load_index(document_folder: str, vector_db_path: str):
    """Builds or loads the FAISS index for PDF documents."""
    embedding_model = GoogleGenerativeAIEmbeddings(model=MODEL_EMBEDDING, google_api_key=GEMINI_API_KEY)

    fingerprint = cache_fingerprint(document_folder, MODEL_EMBEDDING, CHUNK_SIZE, CHUNK_OVERLAP)
    vs = load_cached_index(vector_db_path, embedding_model, fingerprint)
    if vs is not None:
        print(f"[FAISS] Loaded local cache. Total vectors: {vs.index.ntotal}")
        return vs, vs.as_retriever(), vs.index.ntotal

    # Load and chunk PDFs
    pdf_files = sorted(f for f in os.listdir(document_folder) if f.lower().endswith(".pdf"))
    all_docs = []

    for file in pdf_files:
        path = os.path.join(document_folder, file)
        loader = PyPDFLoader(path)
        pages = loader.load()

        print(f"[LOAD] {file}: {len(pages)} pages")

        for p in pages:
            p.metadata["source"] = file

        all_docs.extend(pages)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    documents = text_splitter.split_documents(all_docs)

    # Clean encoding issues
    for d in documents:
        try:
            d.page_content.encode("utf-8")
        except UnicodeEncodeError as e:
            print(f"[WARN] Unicode error in {d.metadata.get('source')}: {e}")
        d.page_content = clean_text(d.page_content)

    global _total_embedding_tokens
    all_chunk_texts = [d.page_content for d in documents]
    count_resp = gemini_client.models.count_tokens(
        model=MODEL_EMBEDDING,
        contents=all_chunk_texts
    )
    _total_embedding_tokens = count_resp.total_tokens

    # Show chunk distribution per PDF
    counter = Counter(d.metadata['source'] for d in documents)
    for pdf_name, num_chunks in counter.items():
        print(f"[SPLIT] {pdf_name}: {num_chunks} chunks created")

    # Build FAISS index
    vectorstore = FAISS.from_documents(documents, embedding_model)
    total_chunks = vectorstore.index.ntotal

    print(f"[FAISS] Created vectors: {total_chunks}")

    os.makedirs(vector_db_path, exist_ok=True)
    save_cached_index(vectorstore, vector_db_path, fingerprint)

    print(f"[FAISS] Saved index to {vector_db_path}")
    return vectorstore, vectorstore.as_retriever(), total_chunks


# Dataset extraction
def extract_datasets_per_pdf(retriever, query: str, total_chunks: int = 100, top_chunks_per_paper: int = 5):
    """Retrieves chunks via RAG and extracts AQA dataset names via LLM."""
    global _sent_calls_count, _total_input_tokens, _total_output_tokens

    k = max(1, total_chunks // 3)
    fetch_k = max(k, total_chunks // 2)

    results = retriever.vectorstore.max_marginal_relevance_search(
        query, k=k, fetch_k=fetch_k, lambda_mult=0.9
    )

    print(f"[RAG] Retrieved results: {len(results)} chunks")

    # Group chunks per PDF
    groups = defaultdict(list)
    for doc in results:
        src = doc.metadata.get("source", "Unknown")
        groups[src].append(doc.page_content)

    paper_datasets = {}

    # Process each PDF
    for source, chunks in groups.items():
        selected_chunks = chunks[:top_chunks_per_paper]
        combined = "\n\n---\n\n".join(selected_chunks)
        context = truncate_context(combined, MAX_CONTEXT_CHARS)

        prompt = (
            "You are an assistant that analyzes full scientific papers.\n"
            "Given the following extracted text from a paper, extract ONLY the official and recognized names of datasets that satisfy the following conditions:\n"
            "(a) The dataset is specifically used for Human Action Quality Assessment (AQA). This condition must be strictly respected.\n"
            "(b) Include only real, unique, and identifiable resources (dataset, database, or data collection).\n"
            "(c) Do NOT write generic descriptions (e.g., 'a large dataset', 'clinical dataset'); if the paper only describes the data generically, leave this field empty.\n\n"
            f"Text:\n{context}\n\n"
            "Output Instructions:\n"
            "- Return valid JSON only, with a single field 'datasets' that is a list of dataset names, e.g., {\"datasets\": [\"FINA09\",\"MTL-AQA\"]}.\n"
            "- If multiple datasets are mentioned, include all of them in the same list.\n"
            "- If no datasets meet these conditions, return an empty list.\n"
        )

        _rate_limit_block_until_allowed()

        backoff = 5
        extracted = []

        # Retry logic
        for attempt in range(4):
            try:
                _sent_calls_count += 1
                resp = claude_client.messages.parse(
                    model=MODEL,
                    max_tokens=1024,
                    temperature=0.0,
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    output_format=ExtractedDatasets,
                )

                _total_input_tokens += resp.usage.input_tokens
                _total_output_tokens += resp.usage.output_tokens

                extracted = resp.parsed_output.datasets or []
                extracted = [s.strip() for s in extracted if s and s.strip()]
                break

            except Exception as e:
                print(f"[LLM ERROR] paper={source} attempt={attempt+1} error={e}")
                time.sleep(backoff)
                backoff *= 2

        paper_datasets[source] = extracted
        print(f"[EXTRACT] {source}: found {len(extracted)} datasets")

    # Prepare output table
    all_pdf_files = sorted(f for f in os.listdir(DOCUMENT_FOLDER) if f.lower().endswith(".pdf"))

    final_list = []
    for pdf in all_pdf_files:
        datasets = paper_datasets.get(pdf, [])
        datasets_str = ";".join(sorted(set(datasets))) if datasets else ""
        final_list.append({"Title": pdf, "Data Source": datasets_str})

    return final_list


# Execution
start_index_time = time.monotonic()
vectorstore, retriever, total_chunks = build_or_load_index(DOCUMENT_FOLDER, VECTOR_DB_PATH)
end_index_time = time.monotonic()
total_index_time_seconds = end_index_time - start_index_time

query = "Human Action Quality Assessment (AQA) datasets"

start_total_time = time.monotonic()
extracted = extract_datasets_per_pdf(retriever, query, total_chunks=total_chunks, top_chunks_per_paper=5)
end_total_time = time.monotonic()
total_extract_time_seconds = end_total_time - start_total_time

total_pipeline_time_seconds = total_index_time_seconds + total_extract_time_seconds

df_results = pd.DataFrame(extracted)
print("[DONE] Extraction completed.")

[LOAD] An et al. - 2024 - Multi-Stage Contrastive Regression for Action Quality Assessment.pdf: 6 pages
[LOAD] Bai et al. - 2022 - Action Quality Assessment with Temporal Parsing Transformer.pdf: 17 pages
[LOAD] Chariar et al. - 2023 - AI Trainer Autoencoder Based Approach for Squat Analysis and Correction.pdf: 16 pages
[LOAD] Chen - 2022 - A Data Mining‐Based Model for Evaluating Tennis Players’ Training Movements.pdf: 11 pages
[LOAD] Chen et al. - 2021 - SportsCap Monocular 3D Human Motion Capture and Fine-Grained Understanding in Challenging Sports Vi.pdf: 19 pages
[LOAD] He et al. - 2022 - Manual Operation Evaluation Based on Vectorized Spatio-Temporal Graph Convolutional for Virtual Reality Training in Smart Grid.pdf: 17 pages
[LOAD] Hipiny et al. - 2023 - Who danced better ranked tiktok dance video dataset and pairwise action quality assessment method.pdf: 12 pages
[LOAD] Ingwersen et al. - 2023 - Video-based Skill Assessment for Golf Estimating Golf Handicap.pdf: 9 pages
[LOAD] 

In [3]:
# --- FINAL REPORT AND COSTS ---

theoretical_min_time = ((_sent_calls_count - 1) // _RATE_MAX_CALLS) * _RATE_WINDOW_SEC if _sent_calls_count > 0 else 0

print("\n" + "="*40)
print("EXECUTION REPORT - DATASET EXTRACTION")
print("="*40)

print(f"Papers Processed: {len(extracted)}")
print(f"API Calls made: {_sent_calls_count}")
print(f"Input Tokens consumed: {_total_input_tokens}")
print(f"Output Tokens generated: {_total_output_tokens}")
print(f"Embedding Tokens: {_total_embedding_tokens}")
print(f"Execution Time: {total_pipeline_time_seconds/60:.2f} min (Theoretical: {theoretical_min_time/60:.2f} min)")

# Pricing
PRICE_PER_1M_INPUT_TOKENS = 3
PRICE_PER_1M_OUTPUT_TOKENS = 15
PRICE_PER_1M_EMBEDDING_TOKENS = 0.20

cost_input = (_total_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_total_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_embedding = (_total_embedding_tokens / 1_000_000) * PRICE_PER_1M_EMBEDDING_TOKENS
cost_total = cost_input + cost_output + cost_embedding

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Embedding Cost: ${cost_embedding:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - DATASET EXTRACTION
Papers Processed: 28
API Calls made: 28
Input Tokens consumed: 73393
Output Tokens generated: 549
Embedding Tokens: 389806
Execution Time: 3.78 min (Theoretical: 2.00 min)
----------------------------------------
Input Cost:  $0.22018
Output Cost: $0.00823
Embedding Cost: $0.07796
Estimated Total Cost: $0.30638


In [4]:
folder_path = data_directory()
result_path = results_directory()
ELIGIBILITY_DIR = result_path + 'CLAUDE_eligibility'

def append_run_summary(filepath, row_dict):
    os.makedirs(os.path.dirname(os.path.abspath(filepath)), exist_ok=True)
    file_exists = os.path.isfile(filepath)
    with open(filepath, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

append_run_summary(
    f"{ELIGIBILITY_DIR}/runs_cost_summary.csv",
    {
        "stage": "eligibility",
        "model": MODEL,
        "records_processed": len(df_results),
        "api_calls": _sent_calls_count,
        "input_tokens": _total_input_tokens,
        "output_tokens": _total_output_tokens,
        "embedding_tokens": _total_embedding_tokens,
        "index_time_sec": total_index_time_seconds,
        "extraction_time_sec": total_extract_time_seconds,
        "execution_time_sec": total_pipeline_time_seconds,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_embedding_usd": cost_embedding,
        "cost_usd": cost_total,
    }
)

In [5]:
# Harmonization

_harm_calls_count = 0
_harm_input_tokens = 0
_harm_output_tokens = 0

def harmonize_all_data_sources(df: pd.DataFrame, client, model=None, max_retries: int = 3):
    """Harmonizes dataset names across all rows using an LLM."""
    global _harm_calls_count, _harm_input_tokens, _harm_output_tokens

    if model is None:
        model = MODEL

    # Prepare records for the harmonization prompt
    records = []
    for idx, row in df.iterrows():
        sources = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        records.append(f"- id: {int(idx)}\n  Data Source: {sources}")

    prompt = (
        "Check and harmonize all dataset names for consistency and correctness.\n"
        "Guidelines:\n"
        "- Remove descriptors like 'dataset', 'database', 'data'.\n"
        "- If multiple datasets are listed, leave the separation with ';' (semicolon) without spaces between the datasets.\n"
        "- Always capitalize the first letter of each dataset.\n"
        "- Prefer the recognized abbreviation if it exists, otherwise use the full name (e.g., 'Massachusetts Institute of Technology' becomes 'MIT','Piano Skills Assessment' becomes 'PISA').\n"
        "- If a field is empty, keep it empty.\n\n"
        "Output Instructions:\n"
        "- Return only JSON matching the schema.\n"
        "- Return exactly one harmonized string per input row, in the same order as provided.\n\n"
        "Records to harmonize:\n"
        + "\n".join(records)
    )

    # Retry mechanism for robustness
    backoff = 5
    for attempt in range(max_retries):
        _rate_limit_block_until_allowed()
        try:
            _harm_calls_count += 1
            resp = claude_client.messages.parse(
                model=model,
                max_tokens=4096,
                temperature=0.0,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                output_format=HarmonizedSources,
            )

            _harm_input_tokens += resp.usage.input_tokens
            _harm_output_tokens += resp.usage.output_tokens

            harmonized = resp.parsed_output.sources

            # Ensure output length matches input length
            if len(harmonized) != len(df):
                harmonized = (harmonized + [""] * len(df))[:len(df)]
            return harmonized

        except Exception as e:
            print(f"[HARMONIZE ERROR] attempt={attempt+1} error={e}")
            time.sleep(backoff)
            backoff *= 2

    # Fallback: return original values unchanged
    fallback = []
    for _, row in df.iterrows():
        s = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        fallback.append(s)

    return fallback


# Apply harmonization to dataframe
start_harm_time = time.monotonic()
df_results["Data Source Harmonized"] = harmonize_all_data_sources(df_results, claude_client, model=MODEL_SUPERVISOR)
end_harm_time = time.monotonic()
total_harm_time_seconds = end_harm_time - start_harm_time

df_results

,Title,Data Source,Data Source Harmonized
0,An et al. - 2024 - Multi-Stage Contrastive Reg...,FineDiving,FineDiving
1,Bai et al. - 2022 - Action Quality Assessment ...,AQA-7;JIGSAWS;MTL-AQA,AQA-7;JIGSAWS;MTL-AQA
2,Chariar et al. - 2023 - AI Trainer Autoencoder...,,
3,Chen - 2022 - A Data Mining‐Based Model for Ev...,,
4,Chen et al. - 2021 - SportsCap Monocular 3D Hu...,AQA;FineGym;MTL-AQA;SMART,AQA;FineGym;MTL-AQA;SMART
5,He et al. - 2022 - Manual Operation Evaluation...,,
6,Hipiny et al. - 2023 - Who danced better ranke...,AIST Dance Video Database;Let's Dance;UID,AIST;Let's Dance;UID
7,Ingwersen et al. - 2023 - Video-based Skill As...,GolfDB;KIMORE,GolfDB;KIMORE
8,Jain et al. - 2021 - Action Quality Assessment...,Fis-V;MIT-Diving;MIT-skate;Multitask AQA Datas...,Fis-V;MIT-Diving;MIT-Skate;MTL-AQA;UNLV-Vault;...
9,Joung et al. - 2023 - Contrastive Learning for...,AI-Hub Fitness Dataset;Squat Dataset,AI-Hub Fitness;Squat


In [6]:
# --- FINAL REPORT AND COSTS (HARMONIZATION) ---

print("\n" + "="*40)
print("EXECUTION REPORT - HARMONIZATION")
print("="*40)

print(f"API Calls made: {_harm_calls_count}")
print(f"Input Tokens consumed: {_harm_input_tokens}")
print(f"Output Tokens generated: {_harm_output_tokens}")

# Pricing Claude Opus 4-5
PRICE_PER_1M_INPUT_TOKENS = 5
PRICE_PER_1M_OUTPUT_TOKENS = 25

cost_input = (_harm_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_harm_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_total = cost_input + cost_output

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - HARMONIZATION
API Calls made: 1
Input Tokens consumed: 957
Output Tokens generated: 303
----------------------------------------
Input Cost:  $0.00478
Output Cost: $0.00758
Estimated Total Cost: $0.01236


In [7]:
append_run_summary(
    f"{ELIGIBILITY_DIR}/runs_cost_summary.csv",
    {
        "stage": "harmonization",
        "model": MODEL_SUPERVISOR,
        "records_processed": len(df_results),
        "api_calls": _harm_calls_count,
        "input_tokens": _harm_input_tokens,
        "output_tokens": _harm_output_tokens,
        "embedding_tokens": None,
        "index_time_sec": None,
        "extraction_time_sec": None,
        "execution_time_sec": total_harm_time_seconds,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_embedding_usd": None,
        "cost_usd": cost_total,
    }
)

In [8]:
df_results.to_csv(f"{ELIGIBILITY_DIR}/records.csv", index=False)

In [10]:
# Final DataFrame Preparation
df_evaluated_final = df_results.copy()
df_evaluated_final["Data Source"] = df_evaluated_final["Data Source Harmonized"]
df_evaluated_final.drop(columns=["Data Source Harmonized"], inplace=True)

# Clean and normalize the Title field
df_evaluated_final["Title"] = (
    df_evaluated_final["Title"]
    .astype(str)
    .str.replace(r"^.* - ", "", regex=True)
    .str.replace(r"\.pdf$", "", flags=re.IGNORECASE, regex=True)
    .str.strip()
)
df_evaluated_final

,Title,Data Source
0,Multi-Stage Contrastive Regression for Action ...,FineDiving
1,Action Quality Assessment with Temporal Parsin...,AQA-7;JIGSAWS;MTL-AQA
2,AI Trainer Autoencoder Based Approach for Squa...,
3,A Data Mining‐Based Model for Evaluating Tenni...,
4,SportsCap Monocular 3D Human Motion Capture an...,AQA;FineGym;MTL-AQA;SMART
5,Manual Operation Evaluation Based on Vectorize...,
6,Who danced better ranked tiktok dance video da...,AIST;Let's Dance;UID
7,Video-based Skill Assessment for Golf Estimati...,GolfDB;KIMORE
8,Action Quality Assessment Using Siamese Networ...,Fis-V;MIT-Diving;MIT-Skate;MTL-AQA;UNLV-Vault;...
9,Contrastive Learning for Action Assessment Usi...,AI-Hub Fitness;Squat


In [11]:
# Merge abstract metadata with the harmonized dataset sources
folder_path = data_directory()
result_path = results_directory()

df_abstract = pd.read_csv(result_path + 'CLAUDE_df_evaluated_validation_abstract.csv')

# Sort both tables by Title for proper alignment
df_abstract_sorted = df_abstract.sort_values(by="Title").reset_index(drop=True)
df_evaluated_final_sorted = df_evaluated_final.sort_values(by="Title").reset_index(drop=True)

# Combine metadata with dataset information
df_review = pd.DataFrame({
    "DOI": df_abstract_sorted["DOI"],
    "Title": df_abstract_sorted["Title"],
    "Authors": df_abstract_sorted["Authors"],
    "Year": df_abstract_sorted["Year"],
    "Include": df_abstract_sorted["Include"],
    "Data Source": df_evaluated_final_sorted["Data Source"]
})

# Remove papers without datasets
df_review = df_review[df_review["Data Source"] != ""].reset_index(drop=True)
df_review = df_review.drop(columns=["Include"], errors="ignore")
df_review

,DOI,Title,Authors,Year,Data Source
0,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,Fis-V;MIT-Diving;MIT-Skate;MTL-AQA;UNLV-Vault;...
1,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,MTL-AQA
2,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,DREAM
3,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,AQA-7;MTL-AQA
4,10.1007/978-3-031-19772-7_25,Action Quality Assessment with Temporal Parsin...,"Bai, Y.; Zhou, D.; Zhang, S.; Wang, J.; DIng, ...",2022,AQA-7;JIGSAWS;MTL-AQA
5,10.1109/ACCESS.2023.3305372,Contrastive Learning for Action Assessment Usi...,C. -I. Joung; S. Byun; S. Baek,2023,AI-Hub Fitness;Squat
6,10.1155/2022/9402195,Design of a Professional Sports Competition Ad...,"Lin, Q.; Zou, J.",2022,CAD-60;MSRC-12;Northwestern-UCLA
7,10.1109/WACV48630.2021.00044,EAGLE-Eye: Extreme-pose Action Grader using de...,M. Nekoui; F. O. Tito Cruz; L. Cheng,2021,AQA-7;ExPose;G-ExPose;MPII
8,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,P. Parmar; B. T. Morris,2017,HMDB51;MIT-Diving;Sports-1M;UCF-101
9,10.1109/CVPR52688.2022.00323,Likert Scoring with Grade Decoupling for Long-...,A. Xu; L. -A. Zeng; W. -S. Zheng,2022,Fis-V;Rhythmic Gymnastics


In [27]:
# Map partial dataset names to standardized identifiers
dataset_map = {
    "AQA": ["AQA-7"],
    "MIT-Skating": ["MIT-Skate"],
    "MIT-Diving": ["MIT-Dive"],
    "MIT-Skiing": ["MIT-Skate"],
    "Olympics Diving": ["MIT-Dive"],
    "MIT Olympic Scoring": ["MIT-Skate", "MIT-Dive"],
    "Olympic sports": ["MIT-Skate", "MIT-Dive"],
    "UNLV-Skating": ["UNLV-Skate"],
    "UNLV-Diving": ["UNLV-Dive"],
    "Vault": ["UNLV-Vault"],
    "Gymnastic vaults": ["UNLV-Vault"],
    "UNLV Olympic Scoring": ["UNLV-Skate", "UNLV-Dive", "UNLV-Vault"],
    "FineDiving-HM": ["FineDiving"],
    "Fis-V": ["FIS-V"],
    "FisV-5": ["FIS-V"],
    "IntelliRehabDS": ["IRDS"],
    "Multitask-AQA": ["MTL-AQA"],
    "Fitness Action": ["Fitness-AQA"],
    "FitnessAQA": ["Fitness-AQA"],
    "RG": ["Rhythmic Gymnastics"],
    "MLT-AQA": ["MTL-AQA"],
    "AFG-Olympics": ["AGF-Olympics"],
    "Epic Skills 2018": ["EPIC-Skills"],
    "MIT-Diving & Skiing": ["MIT-Dive"],
    "Kimore": ["KIMORE"],
    "G-ExPose": ["ExPose"],

    "GolfDB": ["Golf Swing Video"],
    "AIST": ["self-created"],
    "Let's Dance": ["self-created"],
    "UID": ["self-created"]
}

def map_partial_sources(cell):
    """Replace partial or alternative dataset names with standardized identifiers."""
    if pd.isna(cell):
        return cell

    # Split multiple datasets by semicolon
    parts = [p.strip() for p in str(cell).split(";") if p.strip()]
    out = []

    for p in parts:
        matched = False
        for k, v in dataset_map.items():
            pattern = fr"(^|;){re.escape(k)}($|;)"
            if re.fullmatch(k, p):
                out.extend(v)
                matched = True
                break
            elif re.search(pattern, ";" + p + ";"):
                out.extend(v)
                matched = True
                break
        if not matched:
            out.append(p)

    # Remove duplicates and limit length
    seen = set()
    result = []
    for x in out:
        key = x.strip()
        if key not in seen:
            seen.add(key)
            result.append(key)

    result = [x for x in result if len(x) <= 20]
    return ";".join(result)

# Apply mapping to harmonize dataset names
df_review["Data Source"] = df_review["Data Source"].apply(map_partial_sources)
df_review

,DOI,Title,Authors,Year,Data Source
0,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,FIS-V;MIT-Dive;MIT-Skate;MTL-AQA;UNLV-Vault;UN...
1,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,MTL-AQA
2,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,DREAM
3,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,AQA-7;MTL-AQA
4,10.1007/978-3-031-19772-7_25,Action Quality Assessment with Temporal Parsin...,"Bai, Y.; Zhou, D.; Zhang, S.; Wang, J.; DIng, ...",2022,AQA-7;JIGSAWS;MTL-AQA
5,10.1109/ACCESS.2023.3305372,Contrastive Learning for Action Assessment Usi...,C. -I. Joung; S. Byun; S. Baek,2023,AI-Hub Fitness;Squat
6,10.1155/2022/9402195,Design of a Professional Sports Competition Ad...,"Lin, Q.; Zou, J.",2022,CAD-60;MSRC-12;Northwestern-UCLA
7,10.1109/WACV48630.2021.00044,EAGLE-Eye: Extreme-pose Action Grader using de...,M. Nekoui; F. O. Tito Cruz; L. Cheng,2021,AQA-7;ExPose;MPII
8,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,P. Parmar; B. T. Morris,2017,HMDB51;MIT-Dive;Sports-1M;UCF-101
9,10.1109/CVPR52688.2022.00323,Likert Scoring with Grade Decoupling for Long-...,A. Xu; L. -A. Zeng; W. -S. Zheng,2022,FIS-V;Rhythmic Gymnastics


In [28]:
# Load already reviewed data based on abstracts
df_review_abstract = pd.read_csv(result_path + 'CLAUDE_review.csv')
df_review_abstract

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,MMW-AQA
1,10.1109/CVPR52729.2023.00238,LOGO: A Long-Form Video Dataset for Group Acti...,S. Zhang; W. Dai; S. Wang; X. Shen; J. Lu; J. ...,2023,LOGO
2,10.1109/TIM.2024.3398072,Learning Sparse Temporal Video Mapping for Act...,S. Zahan; G. Mubashar Hassan; A. Mian,2024,AGF-Olympics
3,10.1109/CVPR52688.2022.00296,FineDiving: A Fine-grained Dataset for Procedu...,J. Xu; Y. Rao; X. Yu; G. Chen; J. Zhou; J. Lu,2022,FineDiving
4,10.1109/WACV57701.2024.00012,PECoP: Parameter Efficient Continual Pretraini...,A. Dadashzadeh; S. Duan; A. Whone; M. Mirmehdi,2024,JIGSAWS;MTL-AQA;FineDiving;PD4T
...,...,...,...,...,...
61,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,self-created
62,10.1007/978-3-319-10599-4_36,Assessing the quality of actions,"Pirsiavash, H.; Vondrick, C.; Torralba, A.",2014,MIT-Skate;MIT-Dive
63,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,AQA-7;MTL-AQA;JIGSAWS
64,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,self-created


In [29]:
# Combine all review data and export final CSV
df_review_final = pd.concat([df_review, df_review_abstract], ignore_index=True)
df_review_final.to_csv(result_path + 'CLAUDE_review_final.csv', index=False)
df_review_final

,DOI,Title,Authors,Year,Data Source
0,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,FIS-V;MIT-Dive;MIT-Skate;MTL-AQA;UNLV-Vault;UN...
1,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,MTL-AQA
2,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,DREAM
3,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,AQA-7;MTL-AQA
4,10.1007/978-3-031-19772-7_25,Action Quality Assessment with Temporal Parsin...,"Bai, Y.; Zhou, D.; Zhang, S.; Wang, J.; DIng, ...",2022,AQA-7;JIGSAWS;MTL-AQA
...,...,...,...,...,...
82,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,self-created
83,10.1007/978-3-319-10599-4_36,Assessing the quality of actions,"Pirsiavash, H.; Vondrick, C.; Torralba, A.",2014,MIT-Skate;MIT-Dive
84,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,AQA-7;MTL-AQA;JIGSAWS
85,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,self-created


In [30]:
# Compare DOIs between study dataset and validation set
with open(folder_path + 'study_datasets.json', 'r') as f:
    study_papers_list = json.load(f)

verified_dois_study = set(study_papers_list.keys())
verified_dois_validation = set(df_review_final["DOI"].tolist())

matches = verified_dois_study & verified_dois_validation
num_matches = len(matches)

only_study = verified_dois_study - verified_dois_validation
num_study = len(only_study)

only_validation = verified_dois_validation - verified_dois_study
num_validation = len(only_validation)

print("Perfect Matches:", num_matches)
print("Only in Study:", num_study)
print("Only in Validation:", num_validation)

Perfect Matches: 64
Only in Study: 30
Only in Validation: 23


In [31]:
# Compare dataset names between validation and study
filtered_sources = df_review_final['Data Source'].dropna().unique().tolist()
filtered_sources = [source.strip() for item in filtered_sources for source in item.split(';')]
filtered_sources = set(filtered_sources)
filtered_sources.discard("self-created")

# Extract verified datasets from the study JSON
verified_datasets_study = set()
for value in study_papers_list.values():
    if value:
        parts = [v.strip() for v in value.split(';') if v.strip()]
        verified_datasets_study.update(parts)
verified_datasets_study.discard("self-created")

# Compute intersections and differences
common_elements = filtered_sources & verified_datasets_study
only_in_filtered = filtered_sources - verified_datasets_study
only_in_original = verified_datasets_study - filtered_sources


print("Elements in common:", common_elements)
print("Number of elements in common:", len(common_elements))

print("\nElements in Validation but not in Study:", only_in_filtered)
print("Number:", len(only_in_filtered))

print("\nElements in Study but not in Validation:", only_in_original)
print("Number:", len(only_in_original))

Elements in common: {'UMONS-TAICHI', 'FineFS', 'MMASD', 'MIT-Dive', 'Rhythmic Gymnastics', 'AI-Hub Fitness', 'MMW-AQA', 'OlympicFS', 'UNLV-Vault', 'FineDiving', 'IRDS', 'PaSk', 'JIGSAWS', 'EHE', 'FIS-V', 'PD4T', 'MIT-Skate', 'MTL-AQA', 'RFSJ', 'Origami Video', 'Fitness-28', 'PT13', 'Squat', 'FineGym', 'Walking Gait', 'UI-PRMD', 'KIMORE', 'Fri2023', 'LOGO', 'TASD-2', 'PISA', 'AQA-7', 'DREAM', 'TaiChi-24', 'UNLV-Skate', 'JDM', 'UNLV-Dive', 'NETS'}
Number of elements in common: 38

Elements in Validation but not in Study: {'HMDB51', 'UCF-101', 'Sports-1M', 'Northwestern-UCLA', 'ActivityNet1.3', 'CAD-60', 'KTH', 'Thumos14', 'MSRC-12', 'Golf Swing Video', 'ExPose', 'AGF-Olympics', 'RDSD', 'FineRehab', 'MPII', 'SMART', 'Fitness-AQA', 'TAQR', 'MSR-Action3D'}
Number: 19

Elements in Study but not in Validation: {'Stroke Rehabilitation', 'BEST', 'EPIC-Skills', 'TGC20ReId', 'SU-41 Gesture', 'FS1OOO', 'CPR-Coach', 'SkatingVerse', 'Perturbed Workout'}
Number: 9


In [11]:
folder_path = data_directory()
result_path = results_directory()

df_review_final_CLAUDE = pd.read_csv(result_path + "CLAUDE_review_final.csv")
df_review_final_GEMINI = pd.read_csv(result_path + "review_final.csv")

verified_dois_CLAUDE = set(df_review_final_CLAUDE["DOI"].tolist())
verified_dois_GEMINI = set(df_review_final_GEMINI["DOI"].tolist())

matches = verified_dois_CLAUDE & verified_dois_GEMINI
num_matches = len(matches)

only_CLAUDE = verified_dois_CLAUDE - verified_dois_GEMINI
num_CLAUDE = len(only_CLAUDE)

only_GEMINI = verified_dois_GEMINI - verified_dois_CLAUDE
num_GEMINI = len(only_GEMINI)

print("Perfect Matches:", num_matches)
print("Only in CLAUDE:", num_CLAUDE)
print("Only in GEMINI:", num_GEMINI)

# Compare dataset names between CLAUDE and GEMINI
filtered_sources = df_review_final_CLAUDE['Data Source'].dropna().unique().tolist()
filtered_sources = [source.strip() for item in filtered_sources for source in item.split(';')]
filtered_sources = set(filtered_sources)
filtered_sources.discard("self-created")

# Extract verified datasets from the study JSON
verified_datasets_GEMINI = df_review_final_GEMINI['Data Source'].dropna().unique().tolist()
verified_datasets_GEMINI = [source.strip() for item in verified_datasets_GEMINI for source in item.split(';')]
verified_datasets_GEMINI = set(verified_datasets_GEMINI)
verified_datasets_GEMINI.discard("self-created")

# Compute intersections and differences
common_elements = filtered_sources & verified_datasets_GEMINI
only_in_filtered = filtered_sources - verified_datasets_GEMINI
only_in_original = verified_datasets_GEMINI - filtered_sources


print("Elements in common:", common_elements)
print("Number of elements in common:", len(common_elements))

print("\nElements in CLAUDE but not in GEMINI:", only_in_filtered)
print("Number:", len(only_in_filtered))

print("\nElements in GEMINI but not in CLAUDE:", only_in_original)
print("Number:", len(only_in_original))

Perfect Matches: 84
Only in CLAUDE: 3
Only in GEMINI: 7
Elements in common: {'KTH', 'FIS-V', 'UCF-101', 'PT13', 'MMASD', 'Thumos14', 'IRDS', 'TaiChi-24', 'MSRC-12', 'UMONS-TAICHI', 'Rhythmic Gymnastics', 'PISA', 'MTL-AQA', 'UNLV-Dive', 'AGF-Olympics', 'MMW-AQA', 'ExPose', 'NETS', 'Golf Swing Video', 'Fri2023', 'AI-Hub Fitness', 'CAD-60', 'DREAM', 'TAQR', 'JDM', 'FineRehab', 'UNLV-Skate', 'Origami Video', 'ActivityNet1.3', 'Fitness-28', 'FineDiving', 'RDSD', 'FineFS', 'PaSk', 'Walking Gait', 'AQA-7', 'TASD-2', 'Northwestern-UCLA', 'MIT-Dive', 'UI-PRMD', 'OlympicFS', 'PD4T', 'MIT-Skate', 'Fitness-AQA', 'SMART', 'EHE', 'LOGO', 'UNLV-Vault', 'Squat', 'KIMORE', 'MSR-Action3D', 'RFSJ', 'FineGym', 'JIGSAWS'}
Number of elements in common: 54

Elements in CLAUDE but not in GEMINI: {'MPII', 'Sports-1M', 'HMDB51'}
Number: 3

Elements in GEMINI but not in CLAUDE: {'EPIC-Skills', 'Golf Swing Action', 'BEST', 'Diving48', 'Kinetics 700', '3D Animation', 'MM-Fit', 'Infinite Grasp', 'Countix', 'Ranked 